# Session 1: Data Preprocessing

Suppose we want an AI model to predict whether a student will pass a course. To do that we need to collect a **dataset**: a table of observations.

- Each row is one observation, also called a **sample**.
- Each input column is a **feature**: age, city, study hours, attendance.
- The answer we want to predict is the **target**: result.


| Student ID | Age | City       | Attendance (%) | Study Hours | Passed |
|-----------:|----:|------------|---------------:|------------:|--------|
| 101        | 20  | Cairo      | 92             | 3.5         | Yes    |
| 102        | NaN | Giza       | 75             | 1.0         | No     |
| 103        | 21  | cairo      | 88             | 4.0         | Yes    |
| 103        | 21  | cairo      | 88             | 4.0         | Yes    |
| 104        | 19  | Alexandria | 60             | 0.5         | No     |

But before using the dataset to train our model, we need to clean the data. For example, the table above has one age missing, Cairo and cairo represent the same city but are different strings, city names are text (most models expect numerical inputs), and attendance and study hours use different numerical scales (one will cause higher impact on the result than the other when it shouldn't be the case). **Preprocessing** means inspecting, correcting, and transforming raw data into a suitable representation for a model.




## Steps to Preprocess Data

### 1. Explore
Exploration tells us what needs to be fixed, such as missing values, but doesn't touch anything yet. We use a python library called `pandas` to help us read csv files as `DataFrame` objects (table-like objects) which are easy to explore. A `Series` is an object that represents one column in a `DataFrame`.

**Basic DataFrame Operations:**
| Code | Inside the brackets | What it does | Returns |
|---|---|---|---|
| `df['age']` | One column name | Selects the `age` column | A **Series** |
| `df[['age', 'hours']]` | A list of column names | Selects those columns in that order | A **DataFrame** |
| `df[['age']]` | A list containing one column name | Selects `age` while keeping the table structure | A **DataFrame** |
| `df[row_mask]` | A Boolean Series | Keeps rows marked `True`; excludes rows marked `False` | A **DataFrame**, possibly with fewer rows |
| `df[cell_mask]` | A Boolean DataFrame | Keeps cells marked `True`; replaces cells marked `False` with missing values | A **DataFrame** with the same shape when labels match |
| `df[0:2]` | An integer slice | Selects rows at positions 0 and 1; position 2 is excluded | A **DataFrame** |

In [48]:
import pandas as pd

df = pd.read_csv('student_data_dirty.csv') #convert csv file to DataFrame object

print(f"Shape: {df.shape}") #returns a tuple (number of rows, number of columns)
print("\nInfo:")
df.info() #prints info like column names, data types, etc.
print(f"\nDescribe:\n {df.describe()}") #prints a DataFrame the represents statistical summury of df such as each column's count, mean, min, max, etc.
print(f"\nMissing Value Counts:\n {df.isna().sum()}\n") #df.isna() returns a same size boolean DataFrame with True on every cell with a missing value, .sum() sums the True values in each column, the result is a Series with the missing-value counts for each corresponding column in df
print(f"\nDuplicated Rows Counts:\n {df.duplicated().sum()}") #df.duplicated() returns boolean Series with True for every duplicated row, the first occurrence is False by default. .sum() sums the True values.
print(f"\nFrequencies:\n {df['result'].value_counts()}") #returns a Series mapping each distinct value in the results column to its frequency

Shape: (83, 11)

Info:
<class 'pandas.DataFrame'>
RangeIndex: 83 entries, 0 to 82
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   student_id          83 non-null     str    
 1   age                 77 non-null     float64
 2   city                79 non-null     str    
 3   study_hours         78 non-null     str    
 4   attendance          83 non-null     int64  
 5   family_income       79 non-null     str    
 6   education_level     80 non-null     str    
 7   enrollment_date     83 non-null     str    
 8   review_text         83 non-null     str    
 9   result              83 non-null     str    
 10  mostly_empty_notes  3 non-null      str    
dtypes: float64(1), int64(1), str(9)
memory usage: 7.3 KB

Describe:
               age  attendance
count   77.000000   83.000000
mean    24.142857   65.759036
std     26.288738   21.337354
min     -3.000000   35.000000
25%     20.000000   46.00000

### 2. Clean

In [49]:
df.drop_duplicates() #drops duplicated rows and keeps the first occurance
df.dropna(axis=1, thresh=len(df) // 2) #drops columns (axis = 1) with missing values exceeding thresh (in this case half the number of rows)
df['enrollment_date'] = pd.to_datetime(df['enrollment_date'], errors='coerce') #converts dates columns into datetime values, errors='coerce' makes invalid values NaT (like null)

After dropping duplicated rows and columns with a lot of missing values and converting dates to datetime values, we now want fill missing values. We use a python library called `scikit-learn` for that. We could fill missing values with either
1. mean of their corresponding column
2. most frequent value of their corresponding column
3. median of their corresponding column
4. a constant value you choose

In [50]:
from sklearn.impute import SimpleImputer

md_imp = SimpleImputer(strategy='median') #creates an imputer object
df['family_income'] = df['family_income'].str.replace('EGP', '').str.replace(',','').str.strip()
df[['age', 'attendance', 'family_income']] = md_imp.fit_transform(df[['age', 'attendance', 'family_income']])

mf_imp = SimpleImputer(strategy='most_frequent')
df[['city', 'education_level']] = mf_imp.fit_transform(df[['city', 'education_level']])

**Notes:** 
- `fit_transform()` does two things: `fit` caclulates the median (or mean, most frequent value, etc.) and `transform` fills the missing values with it. Originally the function was two different functions.
- `fit_transform()` expects a DataFrame, therefore we use `df[['column name']]` even if we'll only transform one column. The result is then a `NumPy` array (a 2d array) which could then be written back inside the actual df.

Next we detect outliers. Outliers are values that are unsually high or low compared to other values in their corresponding columns. We detect them using the **IQR rule**

$$lb = Q_1 - (1.5 * IQR)$$
$$ub = Q_3 + (1.5 * IQR)$$
$$\text{where } IQR = Q_3 - Q_1 \text{ and } Q_i \text{ is the } i^{\text{th}} \text{ quartile value}$$

Any value outside the $[lb, ub]$ is then flagged. We could then either remove their corresponding row (option A), or replace them with the $lb$ value (in case its lower) or $ub$ value (in case its higher) (option B).

In [51]:
Q1 = df['age'].quantile(0.25)
Q2 = df['age'].quantile(0.75)
IQR = Q1 - Q2
lo = Q1 - 1.5 * IQR
hi = Q2 + 1.5 * IQR

df_A = df[df['age'].between(lo, hi)] #between() returns a boolean Series where values outside [lo, hi] range become False (Option A)
# df = df_A
df['age'] = df['age'].clip(lo, hi) #returns a Series where values below lo become lo and higher than hi become hi (Option B)

In [52]:
Q1 = df['attendance'].quantile(0.25)
Q2 = df['attendance'].quantile(0.75)
IQR = Q1 - Q2
lo = Q1 - 1.5 * IQR
hi = Q2 + 1.5 * IQR

df_A = df[df['attendance'].between(lo, hi)] #between() returns a boolean Series where values outside [lo, hi] range become False (Option A)
# df = df_A
df['attendance'] = df['attendance'].clip(lo, hi) #returns a Series where values below lo become lo and higher than hi become hi (Option B)

### 3. Descretize

Sometimes, broad groups are more useful than exact values. For example, we might not care whether a student attended exactly 3 classes or 6 classes out of 20, all we care about is whether he/she attended most of the classes, half of the classes, or barely attended any class. In this case we *descretize* (or *bin*) the attendance column.

In [53]:
df['attendance_group_A'] = pd.cut(df['attendance'], bins=3, labels=['attended many', 'attended half', 'barely attended'])
#pd.cut aims to divide the values into equal width group, regardless of how many values fall into each group

df['attendance_group_B'] = pd.qcut(df['attendance'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
#pd.qcut aims to divide the values into quartiles where almost equal number of values fall into each group, regardless of the range width

#both functions return Series which could be assigned as a new column in df

### 4. Transform

After cleaning, our data may be accurate and complete but still be represented in a way that is unsuitable for a model.

For example:
|study_hours|income |colour|
|----------:|------:|------|
|5          |10,000 |red   |
|10         |20,000 |blue  |

There are two separate issues:
- The numerical columns use very different scales.
- The colour column contains words rather than numerical inputs.

Transformation changes how we represent these values. Scaling handles the first issue; encoding handles the second.

**1. Scaling Numerical Columns**

Suppose a model compares two people using the differences between their features:
- Study hours differ by 5.
- Income differs by 10,000.
In a distance calculation, the income difference could overwhelm the study-hours difference simply because income uses larger numbers.

Scaling puts numerical columns on more comparable scales. It changes their units, while preserving useful relationships between their values.

There are 3 types of scaling:

**a. StandardScaler:** distance from the mean

$$
x_{\text{scaled}} = \frac{x-\text{mean}}{\text{standard deviation}}
$$

It gives the fitted column a mean of approximately zero and a standard deviation of one. It does not restrict values to a fixed interval.

**b. MinMaxScaler:** position within the range

$$
x_{\text{scaled}} = \frac{x-\min}{\max-\min}
$$

The fitted minimum becomes 0; the fitted maximum becomes 1.

**c. RobustScaler:** distance from the median

$$
x_{\text{scaled}} = \frac{x-\text{median}}{IQR}
$$

In this example we use StandardScaler

In [54]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df[['attendance', 'age']] = scaler.fit_transform(df[['attendance', 'age']]) #fit learns the statistics, transform applies it to produce the new array

**2. Encoding Categories**

Encoding solves a different problem: how do we represent categories numerically?

The choice depends on whether the categories have a meaningful order.

**a. Unordered Categories**

If we want to encode cities, for example, we wouldn't want to give cairo = 1, alexandria = 2, and mansoura = 3 because then the model would interpret the cities as ordered, which we don't want. Therefore we use `OneHotEncoder` which gives each city a channel and makes cairo = [1,0,0], alexandria = [0,1,0], and mansoura = [0,0,1] (Different columns for each channel).

**Note:** If the function `fit()` is runned on a column with only [cairo, alexandria] then it will study the categories based on that, if that same fitted object is then used to transform a column with mansoura, it will give an error. We fix this by initializing the `OneHotEncoder` with `handle_unknown='ignore'`. This will transform the unknow mansoura to [0,0].

In [55]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
df['city'] = df['city'].str.strip().str.lower() #remove white space and make all letters small, to avoid interpreting 'cairo' and 'Cairo' as different categories
cols = ohe.fit_transform(df[['city']]) #takes column(s) and returns 2d NumPy array with the encoding, [[1,0,0], [0,0,1]] for example
col_names = ohe.get_feature_names_out(['city']) #takes the original feature name(s) of column(s) it trained on using fit() and returns the new column names it created in the same order as the encodings as 1d NumPy array

df[col_names] = cols 

**b. Ordered Categories**

If we want to encode education level, for example, we simply can let low=0, high=1, and medium=2, because then the model will correctly interpret them as ordered. We use `OrdinalEncoder` for that.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

oe = OrdinalEncoder(categories=[['low', 'medium', 'high']]) #tell the encoder that low is 0 and high is 2

df['education_level'].str.strip().str.lower()
df[['encoded_education_level']] = oe.fit_transform(df[['education_level']])

**c. Target Column**

Target columns aren't used for calculations, so even if they're unordered we could still encode them with single nuumbers, like pass=1 and fail=0 for example. `LabelEncoder` is used for that. It just produces an integer indentifier for every possible result column value.

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['result_encoded'] = le.fit_transform(df['result']) #it takes and returns a 1d array

**3. Text Vectorization**
